# Part 2.1 — Data audit: do the pipeline's numbers match the filings?

For **NVDA** and **AAPL** this notebook takes line items the pipeline produced in Step 1
(`output/<TICKER>/analysis/`) and checks each one against what the company reported in its
**10-K**, using the structured XBRL data SEC EDGAR publishes for every filing
(`data.sec.gov/api/xbrl/companyfacts`). Every filing value carries the XBRL tag and the
accession number of the 10-K it came from, so each row can be traced back to a specific filing.

The two companies were picked for their different fiscal calendars: NVIDIA's fiscal year ends in
late January (FY2026 = Jan 26 2025 – Jan 25 2026), Apple's in late September.

**Verdicts**

| Verdict | Meaning |
|---|---|
| Match | Equal within rounding |
| Definition difference | Both numbers are right, but they measure different things (e.g. FMP's EBITDA definition) |
| Restated (stock split) | Pipeline shows the split-adjusted figure; the original 10-K shows the pre-split one |
| Market-data input | The metric needs a market price, which no filing contains; checked against filing EPS × market price |
| Mislabeled period | The number is attached to the wrong fiscal year |
| MISMATCH | Differs and none of the above explains it |

Output: `audit/data_audit.csv`.

In [1]:
import json
import datetime as dt
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf

ROOT = Path.cwd().parent if Path.cwd().name == "audit" else Path.cwd()
OUT = ROOT / "FinRobot" / "output"
CACHE = ROOT / "audit" / "data"
CACHE.mkdir(parents=True, exist_ok=True)

# SEC asks automated clients to identify themselves; add a contact email here if requests get blocked.
SEC_HEADERS = {"User-Agent": "GR5398-FinRobot-audit research script"}
COMPANIES = {"NVDA": "0001045810", "AAPL": "0000320193"}

TAGS = [
    "Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", "CostOfRevenue",
    "CostOfGoodsAndServicesSold", "GrossProfit", "SellingGeneralAndAdministrativeExpense",
    "ResearchAndDevelopmentExpense", "OperatingIncomeLoss", "DepreciationDepletionAndAmortization",
    "DepreciationAmortizationAndOther", "DepreciationAndAmortization",
    "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest",
    "InterestExpenseNonoperating", "InterestExpense", "NonoperatingIncomeExpense", "GainLossOnInvestments",
    "InvestmentIncomeInterest", "NetIncomeLoss", "EarningsPerShareBasic", "EarningsPerShareDiluted",
    "CashAndCashEquivalentsAtCarryingValue", "LongTermDebt", "CommercialPaper", "OperatingLeaseLiability",
]
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 250)

## Filing data (SEC EDGAR XBRL) and market prices

Facts are cached in `audit/data/` after the first download; delete the cache to re-fetch.
`filed_value()` returns what a 10-K reported for the fiscal year ending on `end`. With
`which="original"` it takes the first 10-K that reported the period; `"latest"` takes the most recent
one, which picks up restatements such as stock-split adjustments.

In [2]:
def load_sec_facts(ticker: str, cik: str) -> dict:
    cache = CACHE / f"sec_facts_{ticker}.json"
    if not cache.exists():
        r = requests.get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json",
                         headers=SEC_HEADERS, timeout=60)
        r.raise_for_status()
        gaap = r.json()["facts"]["us-gaap"]
        cache.write_text(json.dumps({tag: gaap[tag]["units"] for tag in TAGS if tag in gaap}))
    return json.loads(cache.read_text())


def filed_value(facts: dict, tags: list[str], end: str, instant: bool = False, which: str = "latest"):
    """(value, source) a 10-K reported for the annual period (or balance-sheet date) ending `end`."""
    end_date = dt.date.fromisoformat(end)
    for tag in tags:
        for unit_rows in facts.get(tag, {}).values():
            hits = []
            for r in unit_rows:
                if r.get("form") != "10-K" or r["end"] != end:
                    continue
                if instant:
                    if "start" in r:
                        continue
                elif "start" not in r or not 350 <= (end_date - dt.date.fromisoformat(r["start"])).days <= 380:
                    continue
                hits.append(r)
            if hits:
                hits.sort(key=lambda r: r["filed"])
                r = hits[0] if which == "original" else hits[-1]
                return r["val"], f"{tag} (10-K {r['accn']}, filed {r['filed']})"
    return None, "not found in 10-K XBRL"


def close_on_or_before(ticker: str, end: str) -> tuple[float, str]:
    """Unadjusted closing price on the last trading day on or before `end` (cached)."""
    cache = CACHE / "fy_end_prices.json"
    prices = json.loads(cache.read_text()) if cache.exists() else {}
    key = f"{ticker}@{end}"
    if key not in prices:
        end_ts = pd.Timestamp(end)
        hist = yf.Ticker(ticker).history(start=end_ts - pd.Timedelta(days=10), end=end_ts + pd.Timedelta(days=1),
                                         auto_adjust=False)
        prices[key] = [float(hist["Close"].iloc[-1]), str(hist.index[-1].date())]
        cache.write_text(json.dumps(prices, indent=1))
    return tuple(prices[key])


FACTS = {t: load_sec_facts(t, cik) for t, cik in COMPANIES.items()}
{t: len(f) for t, f in FACTS.items()}

{'NVDA': 23, 'AAPL': 19}

## The audit rows

Each row compares one pipeline value with the value derived from the 10-K. Pipeline amounts are taken
from `financial_metrics_and_forecasts.csv` (the table every text section is generated from) except
net income, which that table does not contain, and EV/EBITDA, which comes from the peer table.
Amounts are in USD billions, margins in %, EPS in USD, multiples in x.

Two definitions matter throughout:

* **EBITDA.** Neither company reports EBITDA, so the filing side is the textbook *operating income + D&A*.
  FMP computes it as *pre-tax income + interest expense + D&A*, which also counts non-operating income.
* **"Contribution profit / margin".** In the pipeline this is revenue − cost of revenue, i.e. **gross** profit,
  so it is checked against `GrossProfit`.

In [3]:
def pct(x) -> float:
    return float(str(x).rstrip("%"))


def audit_company(ticker: str) -> list[dict]:
    f = FACTS[ticker]
    a = OUT / ticker / "analysis"
    fm = pd.read_csv(a / "financial_metrics_and_forecasts.csv").set_index("metrics")
    # Margins are stored as strings like '71.1%' and 'N/A', which makes every column text.
    fm = fm.apply(lambda col: pd.to_numeric(col.astype(str).str.rstrip("%"), errors="coerce"))
    inc = pd.read_csv(a / "income_statement_raw_data.csv")
    km = pd.read_csv(a / "key_metrics_raw_data.csv")
    peer_ev = pd.read_csv(a / "peer_ev_ebitda_comparison.csv", index_col=0)

    # The pipeline labels a fiscal year by the calendar year its period ends in.
    fy_end = {f"{pd.Timestamp(d).year}A": d for d in inc["date"]}
    years = sorted(fy_end)
    y, y_prev, y_oldest = years[-1], years[-2], years[-3]
    end, end_prev = fy_end[y], fy_end[y_prev]
    v = lambda tags, e=end, **kw: filed_value(f, tags, e, **kw)
    B = 1e9

    rev, rev_src = v(["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"])
    rev_prev, rev_prev_src = v(["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"], end_prev)
    cogs, cogs_src = v(["CostOfRevenue", "CostOfGoodsAndServicesSold"])
    sga, sga_src = v(["SellingGeneralAndAdministrativeExpense"])
    rnd, _ = v(["ResearchAndDevelopmentExpense"])
    gp, gp_src = v(["GrossProfit"])
    oi, _ = v(["OperatingIncomeLoss"])
    da, _ = v(["DepreciationDepletionAndAmortization", "DepreciationAmortizationAndOther", "DepreciationAndAmortization"])
    pretax, _ = v(["IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest"])
    int_exp, _ = v(["InterestExpenseNonoperating", "InterestExpense"])
    int_exp = int_exp or 0
    nonop = pretax - oi  # net non-operating income
    ni, ni_src = v(["NetIncomeLoss"])
    eps, eps_src = v(["EarningsPerShareBasic"])
    eps_dil, _ = v(["EarningsPerShareDiluted"])
    cash, _ = v(["CashAndCashEquivalentsAtCarryingValue"], instant=True)
    debt, _ = v(["LongTermDebt"], instant=True)
    cp, _ = v(["CommercialPaper"], instant=True)
    leases, _ = v(["OperatingLeaseLiability"], instant=True)
    ebitda_filing = oi + da
    fmp_ebitda_rebuilt = pretax + int_exp + da
    price, price_date = close_on_or_before(ticker, end)

    rows = []
    def row(metric, year, unit, pipeline, filing, source, tol, if_differs, explanation, pipeline_file="financial_metrics_and_forecasts.csv"):
        rows.append(dict(ticker=ticker, metric=metric, year=year, unit=unit, pipeline=pipeline, filing=filing,
                         tol=tol, if_differs=if_differs, explanation=explanation,
                         pipeline_source=pipeline_file, filing_source=source))

    row("Revenue", y, "USD bn", fm.loc["Revenue", y] / B, rev / B, rev_src, 0.0005, None,
        f"Fiscal year ending {end}.")
    row("Revenue", y_prev, "USD bn", fm.loc["Revenue", y_prev] / B, rev_prev / B, rev_prev_src, 0.0005, None,
        f"Fiscal year ending {end_prev}.")
    row("Cost of Operations", y, "USD bn", fm.loc["Cost of Operations", y] / B, cogs / B, cogs_src, 0.0005, None,
        "Pipeline 'Cost of Operations' = cost of revenue.")
    row("SG&A", y, "USD bn", fm.loc["SG&A", y] / B, sga / B, sga_src, 0.0005, None,
        f"Matches, but the pipeline's opex view stops at SG&A: R&D of ${rnd/B:.1f}bn "
        f"({rnd/rev:.1%} of revenue) appears nowhere in the table the agents see.")
    row("Contribution Margin", y, "%", pct(fm.loc["Contribution Margin", y]), gp / rev * 100, gp_src + " / revenue",
        0.05, None, "'Contribution margin' is computed as (revenue - cost of revenue) / revenue, i.e. gross margin; "
        "the label is misleading but the number is right.")
    row("EBITDA", y, "USD bn", fm.loc["EBITDA", y] / B, ebitda_filing / B, "OperatingIncomeLoss + DepreciationDepletionAndAmortization",
        0.0005, "Definition difference",
        f"FMP EBITDA = pre-tax income + interest expense + D&A = {pretax/B:.3f} + {int_exp/B:.3f} + {da/B:.3f} = "
        f"{fmp_ebitda_rebuilt/B:.3f}bn (reproduces the pipeline value)"
        + (" - the 10-K has no separate interest-expense line, so FMP adds 0" if not int_exp else "")
        + f". It therefore includes net non-operating income of {nonop/B:+.3f}bn (pre-tax income - operating income), "
        "which operating EBITDA excludes.")
    row("EBITDA Margin", y, "%", pct(fm.loc["EBITDA Margin", y]), ebitda_filing / rev * 100, "(OperatingIncomeLoss + D&A) / revenue",
        0.05, "Definition difference", "Follows from the EBITDA definition above.")
    row("Revenue Growth", y, "%", pct(fm.loc["Revenue Growth", y]), (rev / rev_prev - 1) * 100, "revenue rows above",
        0.05, None, f"{y} over {y_prev}.")
    row("EPS (basic)", y, "USD", float(fm.loc["EPS", y]), eps, eps_src, 0.005, None,
        f"The pipeline uses basic EPS; diluted EPS is {eps_dil:.2f}. The table does not say which one it shows.")
    row("Net income", y, "USD bn", inc.loc[inc["date"] == end, "netIncome"].iloc[0] / B, ni / B, ni_src, 0.0005, None,
        "Matches, but net income is only in the raw statement CSV, not in financial_metrics_and_forecasts.csv, so it "
        "is not in the prompt the text sections are generated from: any net-income figure in the prose is unsupported.",
        "income_statement_raw_data.csv")
    row("PE Ratio", y, "x", float(fm.loc["PE Ratio", y]), price / eps, f"price {price:.2f} (Yahoo close {price_date}) / basic EPS",
        0.01, "Market-data input",
        f"Not a filing number. FMP's P/E implies a price of {float(fm.loc['PE Ratio', y]) * eps:.2f} vs the "
        f"{price:.2f} close on {price_date}. Note the report labels this trailing multiple 'P/E (Fwd)'.")
    ev_mcap = km.loc[km["date"] == end, "marketCap"].iloc[0]
    bs = pd.read_csv(a / "balance_sheet_raw_data.csv")
    fmp_leases = bs.loc[bs["date"] == end, "capitalLeaseObligations"].iloc[0]
    ev_filing = ev_mcap + debt + (cp or 0) - cash
    row("EV/EBITDA", y, "x", float(peer_ev.loc[int(y[:4]), ticker]), ev_filing / ebitda_filing,
        "(FMP market cap + LongTermDebt + CommercialPaper - Cash) / (OperatingIncome + D&A)", 0.005, "Definition difference",
        f"FMP counts lease liabilities (${fmp_leases/B:.2f}bn; the 10-K's operating leases alone are "
        f"${leases/B:.2f}bn) as debt and divides by its broader EBITDA; "
        f"with filing debt, cash and operating EBITDA the multiple is {ev_filing/ebitda_filing:.2f}x. Market cap "
        f"(${ev_mcap/B:,.0f}bn) is market data and taken from FMP.", "peer_ev_ebitda_comparison.csv")
    if ticker == "NVDA":
        eps_orig, eps_orig_src = filed_value(f, ["EarningsPerShareBasic"], fy_end[y_oldest], which="original")
        eps_restated, _ = filed_value(f, ["EarningsPerShareBasic"], fy_end[y_oldest], which="latest")
        row("EPS (basic)", y_oldest, "USD", float(fm.loc["EPS", y_oldest]), eps_orig, eps_orig_src, 0.005,
            "Restated (stock split)",
            f"The original 10-K for {y_oldest[:4]} reported {eps_orig:.2f}; after the 10-for-1 split of June 2024 later 10-Ks restate it "
            f"to {eps_restated:.2f}, which is what the pipeline shows. Correct, but a reader comparing with the "
            "original filing will see a 10x gap.")
    # The forecast columns: labelled 2025E, but built on the latest actual year.
    fy2025_end = fy_end.get("2025A")
    rev25, rev25_src = v(["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"], fy2025_end)
    row("Revenue", "2025E", "USD bn", fm.loc["Revenue", "2025E"] / B, rev25 / B, rev25_src + " (FY ending in 2025, actual)",
        0.0005, "Mislabeled period",
        f"'2025E' = {y} revenue x 1.05 = forecast for the year after {y}, not fiscal 2025. Forecast labels are "
        f"hard-coded (generate_financial_analysis.py:166-168) while the base year is the latest actual ({y}), so "
        f"the table shows {y} actuals next to a '2025E' column{' for the same fiscal year' if y == '2025A' else ''}.")
    return rows

In [4]:
rows = [r for t in COMPANIES for r in audit_company(t)]
audit = pd.DataFrame(rows)


def verdict(r) -> str:
    if r["unit"] in ("USD bn",) or r["unit"] == "x" and r["metric"] == "PE Ratio":
        within = abs(r["pipeline"] - r["filing"]) <= r["tol"] * abs(r["filing"])
    else:  # %, EPS, multiples: absolute tolerance
        within = abs(r["pipeline"] - r["filing"]) <= r["tol"]
    if within:
        return "Match"
    return r["if_differs"] or "MISMATCH"


audit["delta"] = audit["pipeline"] - audit["filing"]
audit["delta_pct"] = audit["delta"] / audit["filing"].abs() * 100
audit["verdict"] = audit.apply(verdict, axis=1)
table = audit[["ticker", "metric", "year", "unit", "pipeline", "filing", "delta", "delta_pct", "verdict",
               "explanation", "pipeline_source", "filing_source"]]
table.round({"pipeline": 3, "filing": 3, "delta": 3, "delta_pct": 2}).to_csv(ROOT / "audit" / "data_audit.csv", index=False)
table[["ticker", "metric", "year", "unit", "pipeline", "filing", "delta", "delta_pct", "verdict"]].round(3)

,ticker,metric,year,unit,pipeline,filing,delta,delta_pct,verdict
0,NVDA,Revenue,2026A,USD bn,215.938,215.938,0.000,0.000,Match
1,NVDA,Revenue,2025A,USD bn,130.497,130.497,0.000,0.000,Match
2,NVDA,Cost of Operations,2026A,USD bn,62.475,62.475,0.000,0.000,Match
3,NVDA,SG&A,2026A,USD bn,4.579,4.579,0.000,0.000,Match
4,NVDA,Contribution Margin,2026A,%,71.100,71.068,0.032,0.045,Match
5,NVDA,EBITDA,2026A,USD bn,144.552,133.230,11.322,8.498,Definition difference
6,NVDA,EBITDA Margin,2026A,%,66.900,61.698,5.202,8.431,Definition difference
7,NVDA,Revenue Growth,2026A,%,65.500,65.474,0.026,0.040,Match
8,NVDA,EPS (basic),2026A,USD,4.930,4.930,0.000,0.000,Match
9,NVDA,Net income,2026A,USD bn,120.067,120.067,0.000,0.000,Match


In [5]:
summary = table.groupby(["ticker", "verdict"]).size().unstack(fill_value=0)
summary["total"] = summary.sum(axis=1)
summary

verdict,Definition difference,Match,Mislabeled period,Restated (stock split),total
ticker,,,,,
AAPL,3,9,1,0,13
NVDA,3,9,1,1,14


In [6]:
for _, r in table[table["verdict"] != "Match"].iterrows():
    print(f"[{r.ticker} {r.metric} {r.year}] {r.verdict}: {r.explanation}\n")

[NVDA EBITDA 2026A] Definition difference: FMP EBITDA = pre-tax income + interest expense + D&A = 141.450 + 0.259 + 2.843 = 144.552bn (reproduces the pipeline value). It therefore includes net non-operating income of +11.063bn (pre-tax income - operating income), which operating EBITDA excludes.

[NVDA EBITDA Margin 2026A] Definition difference: Follows from the EBITDA definition above.

[NVDA EV/EBITDA 2026A] Definition difference: FMP counts lease liabilities ($2.94bn; the 10-K's operating leases alone are $2.94bn) as debt and divides by its broader EBITDA; with filing debt, cash and operating EBITDA the multiple is 34.08x. Market cap ($4,542bn) is market data and taken from FMP.

[NVDA EPS (basic) 2024A] Restated (stock split): The original 10-K for 2024 reported 12.05; after the 10-for-1 split of June 2024 later 10-Ks restate it to 1.21, which is what the pipeline shows. Correct, but a reader comparing with the original filing will see a 10x gap.

[NVDA Revenue 2025E] Mislabeled pe

## Findings

1. **The reported line items are transcribed correctly.** All 18 rows that compare a reported figure
   (revenue, cost of revenue, SG&A, gross margin, growth, EPS, net income) or a reproducible market ratio (P/E)
   match the 10-K. FMP is not where the report's problems come from.
2. **EBITDA uses an undisclosed, non-operating definition.** FMP's EBITDA is built from pre-tax income, so it
   carries non-operating items. For NVIDIA that adds $11.1bn of mostly investment gains (+8.5% vs operating
   EBITDA, +5.2pp of margin, and EV/EBITDA 31.4x instead of 34.1x); for Apple the effect is negligible (−0.2%).
   The report never states which definition it uses, and the valuation engine multiplies this EBITDA by a target multiple.
3. **Split restatements are applied silently.** NVIDIA's FY2024 EPS is the split-adjusted $1.21, against the $12.05
   in the original 10-K. That is the right choice, but it is not documented.
4. **The real integrity problem is period labels, not values.** The forecast columns are named `2025E–2027E`
   regardless of the latest actual year. For Apple, fiscal 2025 appears twice (`2025A` = $416.2bn and
   `2025E` = $437.0bn); for NVIDIA, `2025E` is 74% above the actual fiscal 2025. These labels are in the prompt,
   which is why the generated prose calls reported actuals "projections" (see the grounding audit).
5. **What the prompt leaves out matters as much as what is in it.** R&D (8–9% of revenue for both companies) and
   net income never reach the text sections; "contribution margin" is gross margin; EPS is basic without saying so;
   and the report labels the trailing P/E "P/E (Fwd)".